In [1]:
!pip -q install peft sentencepiece

In [2]:
import torch
from torch.nn.utils.rnn import pad_sequence
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, Trainer, TrainingArguments
from peft import LoraConfig, get_peft_model, TaskType

### Load Dataset

In [3]:
alpaca = load_dataset("yahma/alpaca-cleaned")
print(alpaca)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

alpaca_data_cleaned.json:   0%|          | 0.00/44.3M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/51760 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['output', 'input', 'instruction'],
        num_rows: 51760
    })
})


In [4]:
small_data = alpaca["train"].shuffle(seed=42).select(range(2500))
split = small_data.train_test_split(test_size=0.1, seed=42)

train_raw = split["train"]
val_raw = split["test"]

print("Train:", len(train_raw))
print("Val  :", len(val_raw))

Train: 2250
Val  : 250


In [5]:
sample = train_raw[0]

print("Instruction:\n", sample["instruction"])
print("\nInput:\n", sample["input"])
print("\nOutput:\n", sample["output"])

Instruction:
 What is the value of x if 
    x    = y+5,
    y    = z+10,
    z    = w+20,
and  w = 80?


Input:
 

Output:
 Plugging the known value of w into the third given equation, we find that z=100. Plugging z into the second given equation, we find that y=110. Plugging y into the first given equation gives x=115.


### Load Tokenizer

In [6]:
MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"
MAX_LENGTH = 256

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    
tokenizer.padding_side = "right"

print(tokenizer.pad_token)
print(tokenizer.eos_token)

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

<|endoftext|>
<|im_end|>


_preprocess dataset_

In [7]:
def build_messages(example):
    user_text = f"Instruction:\n{example["instruction"].strip()}"
    
    if example["input"].strip():
        user_text = user_text + f"\n\nInput:\n{example["input"].strip()}"
    
    message = [
        {"role": "system", "content": "You are a helpful AI assistant."},
        {"role": "user", "content": user_text},
        {"role": "assistant", "content": example["output"].strip()}
    ]
    return message

_test on one sample_

In [8]:
message = build_messages(sample)

text = tokenizer.apply_chat_template(message, tokenize=False)
print(text)

<|im_start|>system
You are a helpful AI assistant.<|im_end|>
<|im_start|>user
Instruction:
What is the value of x if 
    x    = y+5,
    y    = z+10,
    z    = w+20,
and  w = 80?<|im_end|>
<|im_start|>assistant
Plugging the known value of w into the third given equation, we find that z=100. Plugging z into the second given equation, we find that y=110. Plugging y into the first given equation gives x=115.<|im_end|>



### Tokenize Dataset

In [9]:
def tokenize_dataset(example):
    user_text = f"Instruction:\n{example['instruction'].strip()}"
    
    if example["input"].strip():
        user_text = user_text + f"\n\nInput:\n{example['input'].strip()}"
    
    prompt_messages = [
        {"role": "system", "content": "You are a helpful AI assistant."},
        {"role": "user", "content": user_text}
    ]
    full_messages = [
        {"role": "system", "content": "You are a helpful AI assistant."},
        {"role": "user", "content": user_text},
        {"role": "assistant", "content": example["output"].strip()}
    ]

    prompt_text = tokenizer.apply_chat_template(
        prompt_messages,
        tokenize=False,
        add_generation_prompt=True
    )
    prompt_ids = tokenizer(prompt_text, add_special_tokens=False)["input_ids"]
    
    
    full_text = tokenizer.apply_chat_template(
        full_messages,
        tokenize=False
    )
    full_tokens = tokenizer(
        full_text,
        add_special_tokens=False,
        truncation=True,
        max_length=MAX_LENGTH
    )
    
    input_ids = full_tokens["input_ids"]
    attention_mask = full_tokens["attention_mask"]
    
    labels = input_ids.copy()
    prompt_length = min(len(prompt_ids), len(labels))
    labels[:prompt_length] = [-100] * prompt_length
    
    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "labels": labels
    }

_apply tokenizer on dataset_

In [10]:
train_dataset = train_raw.map(tokenize_dataset, remove_columns=train_raw.column_names)
val_dataset = val_raw.map(tokenize_dataset, remove_columns=val_raw.column_names)

print(train_dataset)
print(val_dataset[0].keys())

Map:   0%|          | 0/2250 [00:00<?, ? examples/s]

Map:   0%|          | 0/250 [00:00<?, ? examples/s]

Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 2250
})
dict_keys(['input_ids', 'attention_mask', 'labels'])


_show one sample_

In [11]:
tokenized_sample = train_dataset[0]

print("Input length:", len(tokenized_sample["input_ids"]))
print("Label length:", len(tokenized_sample["labels"]))
print("First 30 labels:", tokenized_sample["labels"][:30])

Input length: 121
Label length: 121
First 30 labels: [-100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100]


### Collate Function

In [12]:
def collate_fn(batch):
    input_ids = [torch.tensor(item["input_ids"], dtype=torch.long) for item in batch]
    attention_mask = [torch.tensor(item["attention_mask"], dtype=torch.long) for item in batch]
    labels = [torch.tensor(item["labels"], dtype=torch.long) for item in batch]

    
    input_ids = pad_sequence(input_ids, batch_first=True, padding_value=tokenizer.pad_token_id)
    attention_mask = pad_sequence(attention_mask, batch_first=True, padding_value=0)
    labels = pad_sequence(labels, batch_first=True, padding_value=-100)
    
    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "labels": labels
    }

### Load Model

In [13]:
dtype = torch.float16 if torch.cuda.is_available() else torch.float32

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=dtype
)

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/988M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

In [14]:
model.config.use_cache = False
model.enable_input_require_grads()
model.gradient_checkpointing_enable()

print(model.__class__.__name__)

Qwen2ForCausalLM


### Lora Config

In [15]:
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias='none',
    task_type=TaskType.CAUSAL_LM,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"]   
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 2,162,688 || all params: 496,195,456 || trainable%: 0.4359


### Training Arguments

In [18]:
training_args = TrainingArguments(
    output_dir="qwen_lora_alpaca",
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    warmup_steps=0.1,
    lr_scheduler_type="cosine",
    num_train_epochs=2,
    logging_steps=20,
    eval_strategy="steps",
    eval_steps=50,
    save_strategy="steps",
    save_steps=50,
    save_total_limit=2,
    fp16=torch.cuda.is_available(),
    report_to="none",
    remove_unused_columns=False,
    load_best_model_at_end=True
)

In [19]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=collate_fn
)

trainer.train()

Step,Training Loss,Validation Loss
50,1.349470,1.330543
100,1.323675,1.327114
150,1.387269,1.326021
200,1.317559,1.325553
250,1.288404,1.326054


TrainOutput(global_step=282, training_loss=1.3155916937699554, metrics={'train_runtime': 389.3645, 'train_samples_per_second': 11.557, 'train_steps_per_second': 0.724, 'total_flos': 2291650168896000.0, 'train_loss': 1.3155916937699554, 'epoch': 2.0})

### Generate Ans

In [20]:
device = next(model.parameters()).device
model.eval()

def generate_answer(instruction, input_text=""):
    user_text = f"Instruction:\n{instruction.strip()}"
    if input_text.strip():
        user_text = user_text + f"\n\nInput:\n{input_text.strip()}"

    messages = [
        {"role": "system", "content": "You are a helpful AI assistant."},
        {"role": "user", "content": user_text}
    ]

    text = tokenizer.apply_chat_template(
        messages,   
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(text, return_tensors="pt").to(device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=120,
            temperature=0.7,
            top_p=0.9,
            do_sample=True
        )

    new_tokens = outputs[0][inputs["input_ids"].shape[1]:]
    answer = tokenizer.decode(new_tokens, skip_special_tokens=True)
    return answer

In [21]:
result = generate_answer(
    "Explain overfitting in simple words",
    "Use a small example"
)

print(result)

Overfitting is when the model becomes too complex, fitting all the training data rather than just the necessary information to make accurate predictions. In other words, it means that the model has learned to fit every single point on its training set, but doesn't generalize well to new or unseen data.

It's important to note that there are different types of overfitting, such as overfitting due to noise (overfitting where the model performs well on the training data) and overfitting due to complexity (where the model fits so well that it doesn't perform well outside of the training


In [22]:
val_sample = val_raw[0]

print("Instruction:\n", val_sample["instruction"])
print("\nInput:\n", val_sample["input"])
print("\nReal Output:\n", val_sample["output"])
print("\nModel Output:\n", generate_answer(val_sample["instruction"], val_sample["input"]))

Instruction:
 Generate an example of a piece of data that fits the given criteria.

Input:
 A grocery store product with a price between $9 and $10

Real Output:
 Product: Organic Extra Virgin Olive Oil 
Price: $9.99

Model Output:
 One example of a piece of data that meets the given criteria is "The price range for a 2-pack of milk, which typically costs around $6 to $7."
